# **Random Forest on REAL Data**
**Project:** BizFlow360 — ML Early Warning System for Kenyan MSMEs
**Author:** Edusei Mikel Lisamba (Team Lead & ML Integration)

**What this notebook does:** Trains a Random Forest classifier on the unified REAL KNBS dataset to see if an ensemble of decision trees can capture non-linear financial patterns better than the Logistic Regression baseline, specifically aiming to improve the low Recall score.

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report)

base_dir = os.path.abspath('..')
print(f"Base directory: {base_dir}")

Base directory: /home/mikel/BizFlow360/ml_models


# 1. Load the Unified Real Dataset
**What this cell does:** Loads the exact same clean dataset used in the baseline notebook to ensure a 100% fair, apples-to-apples comparison.

In [3]:
data_path = os.path.join(base_dir, 'data', 'unified_msme_modeling_data.csv')
df = pd.read_csv(data_path)

print(f"✅ Loaded unified REAL dataset: {df.shape}")
print(f"\nTarget distribution (0 = Stable, 1 = Distressed):")
print(df['distress_label'].value_counts())

✅ Loaded unified REAL dataset: (15814, 23)

Target distribution (0 = Stable, 1 = Distressed):
distress_label
0    10039
1     5775
Name: count, dtype: int64


# 2. Safety Cleaning & Encoding
**What this cell does:** Applies the exact same KNBS placeholder cleaning and Label Encoding for 'county' and 'sector' as the baseline model.

In [4]:
# 1. Safety clean KNBS placeholders
knbs_placeholders = [-19.11, -19.305, -2.4696, -2.4948, -20.58, -73.5, -74.25, -3.528, -8.82]
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    df[col] = df[col].replace(knbs_placeholders, np.nan)
    df[col] = df[col].fillna(df[col].median())

# 2. Encode Categoricals
le_county = LabelEncoder()
le_sector = LabelEncoder()

df['county_encoded'] = le_county.fit_transform(df['county'])
df['sector_encoded'] = le_sector.fit_transform(df['sector'])

print("✅ Data cleaned and categoricals encoded.")

✅ Data cleaned and categoricals encoded.


# 3. Define Features, Split and Scale
**What this cell does:** Selects the 22 features and performs the exact same 80/20 stratified split and Standard Scaling. (Note: Random Forest doesn't strictly require scaling, but we do it to keep the pipeline identical to the baseline).

In [5]:
features = [
    'county_encoded', 'sector_encoded',
    'male_working_owners', 'female_working_owners',
    'total_monthly_expenses', 'monthly_rent_expense', 'monthly_electricity_expense',
    'monthly_credit_expense', 'monthly_social_responsibility_expense',
    'revenue_last_month', 'normal_monthly_revenue', 'net_income_last_month',
    'stock_value_beginning', 'stock_value_end', 'total_turnover_2015',
    'net_income_margin', 'revenue_change_ratio',
    'business_closed', 'number_closed_establishments',
    'revenue_decline', 'zero_or_missing_net_income', 'low_revenue'
]

X = df[features]
y = df['distress_label']

# EXACT same random_state and stratify as baseline
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Train set: {X_train_scaled.shape} | Test set: {X_test_scaled.shape}")

✅ Train set: (12651, 22) | Test set: (3163, 22)


# 4. Train the Random Forest Model
**What this cell does:** Trains the Random Forest. We use `class_weight='balanced'` to force the model to pay more attention to the minority class (Distressed MSMEs), which should help fix the low Recall problem we saw in the baseline.

In [6]:
print("Training Random Forest on REAL data...")

# class_weight='balanced' helps the model catch more of the distressed businesses
rf_model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42, 
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
print("✅ Random Forest model trained successfully!")

Training Random Forest on REAL data...
✅ Random Forest model trained successfully!


# 5. Evaluate the Model
**What this cell does:** Tests the model on the unseen 20% test set. We are specifically looking to see if the ROC-AUC and Recall have improved compared to the Logistic Regression baseline.

In [7]:
y_pred = rf_model.predict(X_test_scaled)
y_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

print("="*48)
print("  REAL DATA — RANDOM FOREST METRICS")
print("="*48)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print("="*48)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

  REAL DATA — RANDOM FOREST METRICS
Accuracy:  0.6617
Precision: 0.5336
Recall:    0.5844
F1-Score:  0.5579
ROC-AUC:   0.6969

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.71      0.73      2008
           1       0.53      0.58      0.56      1155

    accuracy                           0.66      3163
   macro avg       0.64      0.65      0.64      3163
weighted avg       0.67      0.66      0.66      3163



# 6. Save the Model
**What this cell does:** Saves the trained Random Forest model to the `models/trained` directory.

In [8]:
os.makedirs(os.path.join(base_dir, 'models', 'trained', 'on_real_data'), exist_ok=True)

joblib.dump(rf_model, os.path.join(base_dir, 'models', 'trained', 'on_real_data', 'random_forest.joblib'))

print("✅ Random Forest model saved to ml_models/models/trained/on_real_data/random_forest.joblib")

✅ Random Forest model saved to ml_models/models/trained/on_real_data/random_forest.joblib
